# State Space Model

A State Space Model maps a 1-D input signal $u(t)$ to a N-D latent $x(t)$ before projecting to a 1-D output signal $y(t)$ by the next equation:

$$
x'(t) = Ax(t) + Bu(t) \\
y(t) = Cx(t) + Du(t)
$$

where:

- $u(t)$ is the input.
- $x(t)$ the state vector.
- $y(t)$ the output.
- $A, B, C$ and $D$ are the parameters to learn.

In [5]:
from functools import partial
import jax
import jax.numpy as np
from flax import linen as nn
from jax.nn.initializers import lecun_normal, normal
from jax.numpy.linalg import eigh, inv, matrix_power
from jax.scipy.signal import convolve

In [6]:
if __name__ == "__main__":
    # For this tutorial, construct a global JAX rng key
    # But we don't want it when importing as a library
    rng = jax.random.PRNGKey(1)

$D$ could be seen as an skip connection, so we will omit it. For simplicy we assume input and output are one-dimensional

In [7]:
def random_SSM(rng, N):
    a_r, b_r, c_r = jax.random.split(rng, 3)
    A = jax.random.uniform(a_r, (N, N))
    B = jax.random.uniform(b_r, (N, 1))
    C = jax.random.uniform(c_r, (1, N))
    return A, B, C

## Discrete-time SSM: The Recurrent Representation

In order to compute a discrete input sequence ($u_0, u_1, \dots$), instead of a continues function $u(t)$, the SSM must be discretized by a step size $\Delta$ that represents the resolution of the input. So we sample the continuis signal, hecn $u_k=u(k\Delta)$.

Using bilinear method to discretized SSM converting state matrix $A$ into an approximation $\bar{A}$

$$
\begin{align*}
\bar{A} &= (I - \Delta / 2 \cdot A)^{-1} (I + \Delta / 2 \cdot A) \\
\bar{B} &= (I - \Delta / 2 \cdot A)^{-1} \Delta B \\
\bar{C} &= C
\end{align*}
$$

In [8]:
def discretize(A, B, C, step):
    I = np.eye(A.shape[0])
    BL = inv(I - (step / 2.0) * A)
    Ab = BL @ (I + (step / 2.0) * A)
    Bb = (BL * step) @ B
    return Ab, Bb, C

Now we go to a map of sequence to sequence $u_k \rightarrow y_k$ instead of function to function. And the state equation can be seen as RNN, being $x_k$ the hidden state and $\bar{A}$ as the transition matrix.

$$
\begin{align*}
x_k &= \bar{A} x_{k - 1} + \bar{B} u_k \\
y_k &= \bar{C} x_k
\end{align*}
$$

In [9]:
def scan_SSM(Ab, Bb, Cb, u, x0):
    def step(x_k_1, u_k):
        x_k = Ab @ x_k_1 + Bb @ u_k
        y_k = Cb @ x_k
        return x_k, y_k

    return jax.lax.scan(step, x0, u)

Ensambling

In [10]:
def run_SSM(A, B, C, u):
    L = u.shape[0]
    N = A.shape[0]
    Ab, Bb, Cb = discretize(A, B, C, step=1.0 / L)

    # Run recurrence
    return scan_SSM(Ab, Bb, Cb, u[:, np.newaxis], np.zeros((N,)))[1]

## Tangent: A Mechanics Example

In this example, we consider the forward (horizontal) position $y(t)$ of a mass attached to a wall by a spring and a damper. Over time, an external force $u(t)$ is applied to the mass, causing it to move. The dynamics of the system are determined by three physical parameters: the mass $m$, the spring constant $k$, and the damping (friction) coefficient $b$. The objective is to model how the applied force influences the position of the mass over time.

```text
                Spring (k)         Damper (b)
Wall ┃──/\/\/\/\/\/\/\/───||──────────────■ m
                                           │
                                           ├──→ u(t)
                                           │
                                    y(t) ──┘
```

Hence, the differential equation of the system is:

$$
m \ddot{y}(t) + b \dot{y}(t) + k y(t) = u(t)
$$

Starting from this second-order differential equation, the goal is to rewrite the system in the **state-space form**, which requires a set of **first-order differential equations**. 

This is achieved by defining the state vector as:

$$
x(t)=
\begin{bmatrix}
x_1(t)\\
x_2(t)
\end{bmatrix}
=
\begin{bmatrix}
y(t)\\
\dot{y}(t)
\end{bmatrix},
$$

where $x_1(t)$ represents the position and $x_2(t)$ the velocity. Taking the derivative of the state variables gives

$$
\dot{x}_1(t)=\dot{y}(t)=x_2(t).
$$

Rearranging the original equation to isolate the acceleration,

$$
\ddot{y}(t)=
-\frac{k}{m}y(t)
-\frac{b}{m}\dot{y}(t)
+\frac{1}{m}u(t),
$$

and substituting the state variables yields

$$
\dot{x}_2(t)=
-\frac{k}{m}x_1(t)
-\frac{b}{m}x_2(t)
+\frac{1}{m}u(t).
$$

These two equations completely describe the system dynamics. Notice that each state derivative is expressed as a **linear combination of the current state variables and the external input**. This structure makes it possible to collect the coefficients into matrices.

The coefficients multiplying the state vector

$$
\begin{bmatrix}
x_1(t)\\
x_2(t)
\end{bmatrix}
$$

form the **state matrix** $A$. Specifically, the first equation,

$$
\dot{x}_1=0x_1+1x_2,
$$

contributes the first row

$$
[0 \;\; 1],
$$

while the second equation,

$$
\dot{x}_2=
-\frac{k}{m}x_1
-\frac{b}{m}x_2,
$$

contributes the second row

$$
\left[-\frac{k}{m}\;\;-\frac{b}{m}\right].
$$

Likewise, the coefficients multiplying the input $u(t)$ form the **input matrix** $B$. Since the input does not appear in the first equation but appears with coefficient $\frac{1}{m}$ in the second, we obtain

$$
B=
\begin{bmatrix}
0\\
\frac{1}{m}
\end{bmatrix}.
$$

Finally, the output of interest is the position, i.e., $y(t)=x_1(t)$. Therefore, the **output matrix** simply selects the first state variable,

$$
C=
\begin{bmatrix}
1 & 0
\end{bmatrix}.
$$

Combining these matrices yields the standard state-space representation:

$$
\dot{x}(t)=Ax(t)+Bu(t), \qquad y(t)=Cx(t),
$$

with

$$
A=
\begin{bmatrix}
0 & 1\\
-\frac{k}{m} & -\frac{b}{m}
\end{bmatrix},
\qquad
B=
\begin{bmatrix}
0\\
\frac{1}{m}
\end{bmatrix},
\qquad
C=
\begin{bmatrix}
1 & 0
\end{bmatrix}.
$$

## Verification

To verify the result, substitute the matrices into the state-space equation:

$$
\dot{x}(t)=Ax(t)+Bu(t).
$$

Expanding the matrix multiplication,

$$
\begin{aligned}
\dot{x}(t)
&=
\begin{bmatrix}
0 & 1\\
-\frac{k}{m} & -\frac{b}{m}
\end{bmatrix}
\begin{bmatrix}
x_1(t)\\
x_2(t)
\end{bmatrix}
+
\begin{bmatrix}
0\\
\frac{1}{m}
\end{bmatrix}
u(t)\\[6pt]
&=
\begin{bmatrix}
x_2(t)\\
-\frac{k}{m}x_1(t)-\frac{b}{m}x_2(t)
\end{bmatrix}
+
\begin{bmatrix}
0\\
\frac{1}{m}u(t)
\end{bmatrix}\\[6pt]
&=
\begin{bmatrix}
x_2(t)\\
-\frac{k}{m}x_1(t)-\frac{b}{m}x_2(t)+\frac{1}{m}u(t)
\end{bmatrix},
\end{aligned}
$$

which is exactly the pair of first-order equations derived previously:

$$
\dot{x}_1(t)=x_2(t),
$$

$$
\dot{x}_2(t)=
-\frac{k}{m}x_1(t)
-\frac{b}{m}x_2(t)
+\frac{1}{m}u(t).
$$

Similarly, the output equation

$$
y(t)=Cx(t)
$$

gives

$$
y(t)=
\begin{bmatrix}
1 & 0
\end{bmatrix}
\begin{bmatrix}
x_1(t)\\
x_2(t)
\end{bmatrix}
=x_1(t),
$$

confirming that the system output corresponds to the position of the mass.

Therefore, the state-space representation is mathematically equivalent to the original second-order differential equation, providing a compact matrix formulation that serves as the foundation for modern State Space Models such as S4 and Mamba.


In [11]:
def example_mass(k, b, m):
    A = np.array([[0, 1], [-k / m, -b / m]])
    B = np.array([[0], [1.0 / m]])
    C = np.array([[1.0, 0]])
    return A, B, C

We set $u$ to be a continuos function of $t$

In [12]:
import jax.numpy as jnp

def example_force(t):
    x = jnp.sin(10 * t)
    return jnp.where(x > 0.5, x, 0.0)

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def example_ssm():
    # SSM
    ssm = example_mass(k=40, b=5, m=1)

    # L samples of u(t)
    L = 100
    step = 1.0 / L
    ks = np.arange(L)
    u = example_force(ks * step)

    # Approximation of y(t)
    y = run_SSM(*ssm, u)

    plt.style.use("seaborn-v0_8-paper")

    # Configure axes
    fig, (ax1, ax2, ax3) = plt.subplots(3, figsize=(8, 8))

    ax1.set_title("Force $u_k$")
    ax2.set_title("Position $y_k$")
    ax3.set_title("Object")

    ax1.set_xticks([], [])
    ax2.set_xticks([], [])

    # Fixed axis limits
    ax1.set_xlim(0, L)
    ax1.set_ylim(0, 1.1)

    ax2.set_xlim(0, L)
    ax2.set_ylim(-0.001, 0.016)

    ax3.set_xlim(-0.04, 0.06)
    ax3.set_ylim(-1, 1)
    ax3.set_yticks([])

    # Animated lines
    force_line, = ax1.plot([], [], color="red")
    pos_line, = ax2.plot([], [], color="blue")

    # Moving object
    point, = ax3.plot([], [], "ks", markersize=12)

    def init():
        force_line.set_data([], [])
        pos_line.set_data([], [])
        point.set_data([], [])
        return force_line, pos_line, point

    def update(frame):
        force_line.set_data(ks[:frame], u[:frame])
        pos_line.set_data(ks[:frame], y[:frame, 0])

        point.set_data([y[frame, 0]], [0])

        return force_line, pos_line, point

    ani = FuncAnimation(
        fig,
        update,
        frames=range(1, L),
        init_func=init,
        interval=50,
        blit=True,
    )

    plt.close(fig)
    return HTML(ani.to_jshtml())

In [14]:
example_ssm()

## Training SSMs: The Convolutional Representation

In order to do a parallel training, we need to change the representation from RNN to Convolutional one.

Starting from the discrete state-space model

$$
x_{k+1}=\bar{A}x_k+\bar{B}u_k,
\qquad
y_k=\bar{C}x_k,
$$

and assuming

$$
x_{-1}=0,
$$

unrolling the recurrence yields

$$
\begin{aligned}
x_0 &= \bar{B}u_0,\\
x_1 &= \bar{A}\bar{B}u_0+\bar{B}u_1,\\
x_2 &= \bar{A}^2\bar{B}u_0+\bar{A}\bar{B}u_1+\bar{B}u_2,\;\ldots
\end{aligned}
$$

which produces the outputs

$$
\begin{aligned}
y_0 &= \bar{C}\bar{B}u_0,\\
y_1 &= \bar{C}\bar{A}\bar{B}u_0+\bar{C}\bar{B}u_1,\\
y_2 &= \bar{C}\bar{A}^2\bar{B}u_0+\bar{C}\bar{A}\bar{B}u_1+\bar{C}\bar{B}u_2,\;\ldots
\end{aligned}
$$

Recognizing the repeated coefficients,

$$
K_i=\bar{C}\bar{A}^i\bar{B},
$$

defines the **SSM convolution kernel**

$$
\bar{K}
=
(
\bar{C}\bar{B},
\bar{C}\bar{A}\bar{B},
\ldots,
\bar{C}\bar{A}^{L-1}\bar{B}
),
$$

allowing the recurrence to be written compactly as the discrete convolution

$$
y=\bar{K}*u.
$$

Thus, the convolution formulation is mathematically equivalent to the original state-space recurrence while enabling efficient parallel computation.


In [15]:
def K_conv(Ab, Bb, Cb, L):
    return np.array(
        [(Cb @ matrix_power(Ab, l) @ Bb).reshape() for l in range(L)]
    )

For applying this Kernel, we used the non-circular convolutions, we dont get into details, but know just focused in the output

In [16]:
def causal_convolution(u, K, nofft=False):
    if nofft:
        return convolve(u, K, mode="full")[: u.shape[0]]
    else:
        assert K.shape[0] == u.shape[0]
        ud = np.fft.rfft(np.pad(u, (0, K.shape[0])))
        Kd = np.fft.rfft(np.pad(K, (0, u.shape[0])))
        out = ud * Kd
        return np.fft.irfft(out)[: u.shape[0]]

we compare RNN and CNN representations:

In [17]:
def test_cnn_is_rnn(N=4, L=16, step=1.0 / 16):
    ssm = random_SSM(rng, N)
    u = jax.random.uniform(rng, (L,))
    jax.random.split(rng, 3)
    # RNN
    rec = run_SSM(*ssm, u)

    # CNN
    ssmb = discretize(*ssm, step=step)
    conv = causal_convolution(u, K_conv(*ssmb, L))

    # Check
    assert np.allclose(rec.ravel(), conv.ravel())

In [18]:
test_cnn_is_rnn()